In [1]:
import pandas as pd
DATA_PATH = "data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip"

df = pd.read_csv(DATA_PATH, sep="\t")
print("Default settings:", df.shape)

Default settings: (144724, 15)


144,724 rows and 15 columns

In [2]:
df = pd.read_csv(DATA_PATH, sep="\t", quoting=3)  # 3 = treat quotes as plain text
print("quoting=3:", df.shape)

quoting=3: (145431, 15)


144,724 rows and quoting=3 gave 145,431 rows, a difference of 707 rows
Code can run without errors and still load the wrong data. You should check row counts against what you expect and look at the actual data before trusting any analysis built on it.

In [3]:
peek = df["review_body"].sample(5, random_state=2134)
for text in peek:
    print(text, "\n")

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable<br />the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.<br /><br />I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.<br /><br />So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They dont care once they ha

The reviews contain HTML code instead of normal text: `<br />` tags where line breaks should be, `&#34;` instead of quote marks, and extra backslashes like `\\"seems\\"`. There are also typos ("payed").
The reviews were typed into Amazon's website, which stored them as HTML. When the data was exported to a file, the line breaks and quote marks stayed in their HTML and escaped forms.

In [4]:
df["star_rating"].value_counts(normalize=True).sort_index()

star_rating
1    0.171697
2    0.053283
3    0.079962
4    0.140314
5    0.554744
Name: proportion, dtype: float64

The ratings are heavily skewed toward 5 stars: 5 stars = 55.5%, 4 stars = 14.0%, 3 stars = 8.0%, 2 stars = 5.3%, 1 star = 17.2%. About 70% of reviews are positive.

This matters because a model that always guessed "positive" would score about 76% accuracy (leaving out 3-star reviews) without understanding anything. High accuracy on this data can be misleading, and the model could be weak on negative reviews without it showing in the overall number.

In [5]:
df["product_title"].value_counts().head(10)

product_title
Playstation Network Card                                          13642
Xbox Live Subscription                                             7307
Playstation Plus Subscription                                      4712
Xbox Live Gift Card                                                3438
SimCity - Limited Edition                                          3421
Xbox 360 Live Points Card                                          3036
Battlefield 4                                                      1453
Final Fantasy XIV: A Realm Reborn                                  1273
Battlefield 3                                                      1160
Command and Conquer The Ultimate Collection [Online Game Code]      875
Name: count, dtype: int64

Most of them are not games. Five of the top six are gift cards and subscriptions (PlayStation Network Card, Xbox Live Subscription, PlayStation Plus, Xbox Live Gift Card, Xbox 360 Points Card). Reviews of these are about whether the code arrived and worked, not about gameplay, so the dataset is less about "video games" than its name suggests.

In [6]:
print("Reviews with <br />:", df["review_body"].str.contains("<br />").sum())
df = df.dropna(subset=["review_body"])
df["review_body"] = df["review_body"].str.replace("<br />", " ")
print("After cleaning:", df["review_body"].str.contains("<br />").sum(), "\n")

for text in df.loc[peek.index, "review_body"]:
    print(text, "\n")

Reviews with <br />: 25699
After cleaning: 0 

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.  I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.  So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They 

25,699 reviews had `<br />` before cleaning and 0 after
The `&#34;` codes and the backslashes around quotes are still there, along with double spaces and typos ("payed", "Dose"). I don't think this would change most predictions, because the words that carry the sentiment ("Perfect", "crapware", "screwed") are still intact. It adds some noise, but it shouldn't flip a review from positive to negative.

In [7]:
sample = df.groupby("star_rating").sample(n=100, random_state=42)
complaints = df[df["star_rating"] == 1].sample(n=20, random_state=42)
print(len(sample), len(complaints))

500 20


Because 55% of reviews are 5 stars, a random sample of 500 would have about 277 five-star reviews and only about 27 two-star reviews. That is too few low and middle ratings to judge the model on them, and the accuracy would mostly reflect 5-star reviews. Taking 100 of each rating tests the model equally on every rating.

In [8]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)
print(sample["star_rating"].iloc[0], sample["review_body"].iloc[0][:200])
classifier(sample["review_body"].iloc[0])

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

C:\Users\Charles\OneDrive - Babson College\Desktop\hf-explore\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Charles\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Charles\OneDrive - Babson Colle

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

1 Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get 


[{'label': 'NEGATIVE', 'score': 0.9998071789741516}]

The keys are `label` (POSITIVE or NEGATIVE) and `score`. The score is the model's confidence in that label, from 0 to 1. Here it was 0.9998, so the model was almost certain.
Yes. The customer gave 1 star and the model labeled the review NEGATIVE.

In [20]:
%%time
results = classifier(sample["review_body"].tolist(), truncation=True)
sample["label"] = [r["label"] for r in results]
sample["score"] = [r["score"] for r in results]
sample[["star_rating", "label", "score"]].head(10)

CPU times: total: 6min 18s
Wall time: 56 s


,star_rating,label,score
40836,1,NEGATIVE,0.999807
85497,1,NEGATIVE,0.990978
124434,1,NEGATIVE,0.998919
118528,1,NEGATIVE,0.999704
13480,1,NEGATIVE,0.999466
47179,1,NEGATIVE,0.999234
130193,1,NEGATIVE,0.999613
126902,1,NEGATIVE,0.999815
58985,1,NEGATIVE,0.999799
22608,1,NEGATIVE,0.999736


500 reviews took about 30 seconds so 2.4 hours.
The model can only read about 512 tokens at a time. Without truncation, a very long review would cause an error and stop the whole batch. With truncation, long reviews are cut off at the limit so the code keeps running. The trade-off is that the model only reads the start of long reviews.

In [10]:
def stars_to_sentiment(stars):
    """Convert a star rating into the sentiment the customer expressed."""
    if stars <= 2:
        return "NEGATIVE"
    if stars >= 4:
        return "POSITIVE"
    return "NEUTRAL"


sample["truth"] = sample["star_rating"].apply(stars_to_sentiment)
graded = sample[sample["truth"] != "NEUTRAL"]
print("Accuracy:", (graded["label"] == graded["truth"]).mean())

Accuracy: 0.8775


3-star reviews are neutral or mixed, so there is no correct answer for a model that can only say POSITIVE or NEGATIVE. 
The accuracy is 87.75% which is much better than the 50% you would get by guessing on this balanced sample, but about 1 in 8 reviews is still wrong. I would trust it for overall trends across many reviews, but not to judge individual reviews or set star ratings automatically.

In [11]:
pd.crosstab(sample["star_rating"], sample["label"])

label,NEGATIVE,POSITIVE
star_rating,,
1,96,4
2,95,5
3,66,34
4,23,77
5,17,83


4-star reviews: 23 of 100 were labeled NEGATIVE. 5-star reviews were next with 17 wrong. The model does much better on negative reviews (only 4 and 5 wrong for 1 and 2 stars).
It had to pick one side. It labeled 66 of them NEGATIVE and 34 POSITIVE. Mixed reviews usually include complaints, and the model tends to lean negative when it sees them.

In [12]:
mistakes = graded[graded["label"] != graded["truth"]]
for review in mistakes.head(5).itertuples():
    print(review.star_rating, review.label, round(review.score, 2))
    print(review.review_body[:300], "\n")

1 POSITIVE 0.99
Don't get me wrong, the game concept is awesome. Graphics are great.  BUT  If you don't want to lose your progress when the Simcity Servers stop responding, don't buy this. There is no option to save locally just saved to their server so the smallest error will lose your work.  Overall it's a good g 

1 POSITIVE 1.0
Enjoys game 

1 POSITIVE 0.73
I wasted $7. im sure its a good game but i cant even get to the game part. i tried. a lot. 

1 POSITIVE 1.0
Dose not work right on PSVITA so was not happy but it is a fun game love twisted metal games. 

2 POSITIVE 0.87
I bought this game for my 8 year old daughter who likes the kingdom of keflings game on xbox. The graphics are very simple and it is a basic strategy/building style game. Unfortunately, it looks like it would have been cutting edge 15 to 20 years ago, but it really didn't entertain my kids for more  



1. SimCity (1 star, POSITIVE 0.99):Mixed review. It praises the concept and graphics and says "Overall it's a good game," but the server problem was a dealbreaker for the customer. The model followed the positive words. Partly both.
2. "Enjoys game" (1 star, POSITIVE 1.0): The customer's rating is misleading. The text is clearly positive, so they probably clicked the wrong star. The model was right about the text.
3. "I wasted $7..." (1 star, POSITIVE 0.73): Model error. It picked up "good game" but missed that the customer could not get the game to work. Its lower confidence shows it was unsure.
4. PS Vita / Twisted Metal (1 star, POSITIVE 1.0): Mixed review that ends on "fun game love twisted metal games." The model weighted the positive ending, while the rating reflects the technical problem. Partly both.
5. Keflings (2 stars, POSITIVE 0.87):** Model error. The criticism is polite and indirect ("would have been cutting edge 15 to 20 years ago"), and the model misses that kind of complaint.

In [13]:
star_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)
results = star_model(sample["review_body"].tolist(), truncation=True)
sample["pred_stars"] = [int(r["label"][0]) for r in results]  # "4 stars" -> 4

print("Exact:", (sample["pred_stars"] == sample["star_rating"]).mean())
off_by = (sample["pred_stars"] - sample["star_rating"]).abs()
print("Within one star:", (off_by <= 1).mean())

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

C:\Users\Charles\OneDrive - Babson College\Desktop\hf-explore\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Charles\.cache\huggingface\hub\models--nlptown--bert-base-multilingual-uncased-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Exact: 0.574
Within one star: 0.912


Exact star rating: 57.4% of the time. Within one star: 91.2% of the time. Random guessing would get only 20% exact.
The star model It gives a 1 to 5 rating, which matches how publishers already track reviews, and it can show neutral and mixed reviews instead of forcing them into positive or negative. It is within one star 91% of the time, which is enough for tracking trends. The POSITIVE/NEGATIVE model's 88% accuracy only counts the easy reviews; it had to leave out 3-star reviews and can't show anything in between.

In [14]:
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["price", "bugs or crashes", "account or login", "gameplay", "download or install"]
results = zsc(complaints["review_body"].tolist(), candidate_labels=labels)
complaints["topic"] = [r["labels"][0] for r in results]
complaints["topic"].value_counts()

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

C:\Users\Charles\OneDrive - Babson College\Desktop\hf-explore\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Charles\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

topic
download or install    5
bugs or crashes        5
account or login       4
price                  3
gameplay               3
Name: count, dtype: int64

Download or install (5) and bugs or crashes (5) tied for the top spot, followed by account or login (4), price (3) and gameplay (3). 14 of the 20 complaints are technical: customers are mostly unhappy that they can't get the product to work, not with the game itself.
1. Price: Partly. The review mentions paying $30 extra, but it also complains about the interface, the stories and the graphics, so one label misses most of the complaint.
2. Download or install: Yes. The customer installed the game but can't get it to run. "Bugs or crashes" would also fit.
3. Gameplay: Yes. The complaint is about slow pacing, which is a gameplay issue. The model handled the sarcasm well.


In [15]:
for review in complaints.head(3).itertuples():
    print("Topic:", review.topic)
    print(review.review_body[:300], "\n")

Topic: price
Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get VC yea! not). 

Topic: download or install
I too installed and tried to play Cobi Treasure Island. My PC does meet the proper requirements so I do not know what the problem is. 

Topic: gameplay
If you liked \\"Heavy Rain\\", you'll like this.  However, if you'd like for your games to have a faster pace than 'the English Patient', you'll want to skip this title in favor of something more exciting, like \\"Parking Simulator 4\\". 



In [16]:
more_labels = labels + ["technical problems", "HR policy"]
new_results = zsc(complaints["review_body"].head(5).tolist(), candidate_labels=more_labels)
for old, new in zip(results[:5], new_results):
    print("Before:", old["labels"][0], round(old["scores"][0], 2))
    print("After: ", new["labels"][0], round(new["scores"][0], 2), "\n")

Before: price 0.66
After:  price 0.59 

Before: download or install 0.41
After:  download or install 0.3 

Before: gameplay 0.36
After:  gameplay 0.32 

Before: bugs or crashes 0.53
After:  technical problems 0.7 

Before: download or install 0.79
After:  download or install 0.45 



Four of the five reviews kept the same top label, but the scores dropped, some by a lot (0.79 to 0.45, 0.41 to 0.30). Review 4 switched from "bugs or crashes" (0.53) to the new label "technical problems" (0.70).

Each review has a fixed total of 1 to split across the labels. Adding labels means each one gets a smaller share. The overlapping label "technical problems" took share from "bugs or crashes" because they mean almost the same thing. Even the irrelevant label "HR policy" took a small share.

Labels should be distinct, with no overlap, and should match real complaint types. The scores show how well a label fits compared with the other labels, not how confident the model is overall, so changing the label list changes the results. Labels should be tested on real reviews before you trust a dashboard built on them.